# Capítulo 9 — Outros Aspectos de Classificação

**Livro:** começa na página **187** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
set.seed(42)
options(digits = 4, repr.plot.width = 7, repr.plot.height = 4.5)
suppressPackageStartupMessages({ library(glmnet); library(rpart); library(randomForest); library(FNN); library(e1071) })
cat("Pronto.", R.version.string, "
")

## Assimetria na Função de Perda, Conjuntos de Dados Desbalanceados e Outros Cortes · PDF p. 187

#### Exercício: Escolhendo o corte pelo custo

Num problema simulado, errar um positivo (FN) custa 5 e errar um negativo (FP) custa 1. Com uma logística, calcule o custo médio no teste para cortes de 0,05 a 0,95 e ache o melhor. Compare com 1/(1+5).

In [ ]:
gera <- function(n) { x <- rnorm(n); data.frame(x, y = rbinom(n, 1, plogis(-1 + 2 * x))) }
tr <- gera(1000); te <- gera(5000)
p <- predict(glm(y ~ x, binomial, tr), te, type = "response")
cortes <- seq(0.05, 0.95, 0.01)
custo <- sapply(cortes, function(c) mean(5 * (p < c & te$y == 1) + 1 * (p >= c & te$y == 0)))
c(melhor_corte = cortes[which.min(custo)], teoria = 1 / (1 + 5))

## Combinando Classificadores · PDF p. 195

#### Exercício: Voto da maioria

No problema dos círculos, combine árvore, KNN (k = 15) e SVM radial pelo voto da maioria. A combinação ganha de cada um sozinho?

In [ ]:
gera <- function(n) { x <- runif(n, -1, 1); y <- runif(n, -1, 1); data.frame(x, y, classe = factor(x^2 + y^2 < 0.5)) }
tr <- gera(300); te <- gera(2000)
P <- data.frame(arvore = as.character(predict(rpart(classe ~ ., tr), te, type = "class")),
                knn = as.character(FNN::knn(tr[1:2], te[1:2], tr$classe, k = 15)),
                svm = as.character(predict(svm(classe ~ ., tr), te)))
voto <- apply(P, 1, function(v) names(which.max(table(v))))
sapply(cbind(P, voto = voto), function(p) mean(p == as.character(te$classe)))